In [ ]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.run_config import RunConfig
from google.genai import types
from utils.tool import check_warehouse_availability,reserve_warehouse_items


In [16]:
from dotenv import load_dotenv
import os
load_dotenv()

True

### ADK AGENT

In [17]:
model=LiteLlm(
    model="openai/gpt-4.1-mini",
    temperature=0.0,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [18]:
warehouse_agent=Agent(
    name="Warehouse_manger_Agent",
    model=model,
    tools=[check_warehouse_availability,reserve_warehouse_items],
    description="The user is asking items from the Warehouse",
    instruction="""
    You are a part of the shopping assistant that can manage the user's shopping cart.
    ## Instructions
    - Use names specificaly provided in the available tools. Don't add any additional text to the names.
    - You can run multipple tools at once.
    - As the final answer you should return an answer in a form of actions performed.
    """
)

### ADK SESSION

In [19]:
session_service=InMemorySessionService()

In [20]:
await session_service.create_session(
    app_name="warehouse_app",
    user_id="1234",
    session_id="session_1"
    )


Session(id='session_1', app_name='warehouse_app', user_id='1234', state={}, events=[], last_update_time=1788009791.8198726)

### Define Runner

In [21]:
runner=Runner(
    agent=warehouse_agent,
    session_service=session_service,
    app_name="warehouse_app"
)

In [22]:
message=types.Content(
    role="User",
    parts=[types.Part(text="what is the availability of B0B11P5XHV in all of the Warehouses??")]
    
)

In [23]:
message

Content(
  parts=[
    Part(
      text='what is the availability of B0B11P5XHV in all of the Warehouses??'
    ),
  ],
  role='User'
)

In [24]:
result=runner.run(
    user_id="1234",
    session_id="session_1",
    new_message=message,
    run_config=RunConfig(
        max_llm_calls=3,
    )
    
)

In [25]:
for event in result:
    print("=====================")
    print(event)

model_version='gpt-4.1-mini-2025-04-14' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'items': [
            {<... 2 items at Max depth ...>},
          ]
        },
        id='call_Jb7y9KyKzHYjBwy8DsPoKFv9',
        name='check_warehouse_availability'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None turn_complete_reason=None interaction_status=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=33,
  prompt_token_count=452,
  total_token_count=485
) live_session_resumption_update=None live_session_id=None go_away=None voice_activity=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None environment_id=None invocati

In [26]:
async def ask_warehouse_agent(query:str, session_id:str, user_id:str="1234"):
    existing_session=await session_service.get_session(
        app_name="warehouse_app",
        user_id=user_id,
        session_id=session_id
    )

    if not existing_session:
        await session_service.create_session(
            app_name="warehouse_app",
            user_id=user_id,
            session_id=session_id
        )

    runner=Runner(
       agent=warehouse_agent,
       session_service=session_service,
       app_name="warehouse_app"
    )  

    content=types.Content(
       role="User",
       parts=[types.Part(text=query)]
    )

    final_text=""
    
    for event in runner.run(
        user_id=user_id,
        session_id=session_id,
        new_message=content,
        run_config=RunConfig(
            max_llm_calls=3,
        ),
    ):
       if event.is_final_response():
         if event.content and event.content.parts:
            for part in event.content.parts:
                final_text+=part.text
            break

    return final_text or "[ No final response produced ]"


In [27]:
answer=await ask_warehouse_agent("what is the availability of B0B11P5XHV in all of the Warehouses??",session_id="123")

In [29]:
print(answer)

The product B0B11P5XHV is available for complete fulfillment in the following warehouses:
- Berlin Distribution Center (Berlin, Germany) with 77 units available
- Lyon Regional Warehouse (Lyon, France) with 65 units available
- Marseille Mediterranean Hub (Marseille, France) with 63 units available
- Hamburg North Warehouse (Hamburg, Germany) with 55 units available

It is not available in the Munich Logistics Hub and Paris Central Depot.


In [30]:
answer=await ask_warehouse_agent("can u reserve 6 in Lyon",session_id="123")

In [31]:
answer

'I have reserved 6 units of B0B11P5XHV in the Lyon Regional Warehouse.'